# kernel-set — Colab benchmark (L4 / A100)

Benchmarks **every kernel** against a PyTorch reference on a Colab GPU. Auto-detects the GPU and
builds `libkernel_set` for the right CUDA arch (**L4 → sm_89**, **A100 → sm_80**, H100 → sm_90).

**Set a GPU runtime first:** *Runtime → Change runtime type → L4 or A100*.


## 1 · Inspect the GPU


In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import subprocess
cc = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0]
name = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip()
print(f'\nGPU: {name}  cc={cc}  ->  kernel-set builds for sm_{cc.replace(".","")}')


## 2 · Get the source

Set `KS_REPO_URL` to clone your fork, **or** upload the repo and point `KS_REPO_DIR` at it.


In [ ]:
import os
KS_REPO_URL = ''                     # e.g. 'https://github.com/<you>/kernel-set.git'
KS_REPO_DIR = '/content/kernel-set'  # where the repo lives / will be cloned
%cd /content
if KS_REPO_URL and not os.path.isdir(KS_REPO_DIR):
    !git clone --depth 1 $KS_REPO_URL $KS_REPO_DIR
assert os.path.isdir(KS_REPO_DIR), (
    f'{KS_REPO_DIR} not found — set KS_REPO_URL to clone, or upload the repo there.')
%cd $KS_REPO_DIR
!ls


## 3 · Build + benchmark

`build_and_bench.sh` maps the GPU to a CUDA arch, builds the lib for that single arch, points the
Python binding at it (`KERNEL_SET_LIB`), runs `bench.py`, and writes `benchmarks/results/<gpu>.md`.
Tunables: `KS_DTYPE` (fp16/bf16), `KS_OPS` (`all` or comma list), `KS_ITERS`.


In [ ]:
!cmake --version >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y cmake)
import os
os.environ['KS_DTYPE'] = os.environ.get('KS_DTYPE','fp16')
os.environ['KS_OPS']   = os.environ.get('KS_OPS','all')
os.environ['KS_ITERS'] = os.environ.get('KS_ITERS','50')
!bash benchmarks/build_and_bench.sh


## 4 · Results table


In [ ]:
import glob, os
from IPython.display import Markdown, display
reports = sorted(glob.glob('benchmarks/results/*.md'), key=os.path.getmtime)
assert reports, 'No report produced — check the build log above.'
print('report:', reports[-1])
display(Markdown(open(reports[-1]).read()))


## 5 · Which kernels would a model use?

`ksctl plan` auto-selects the strongest kernel per op for `(model, gpu, dtype)`.


In [ ]:
!python3 models/ksctl plan --model llama-3-8b --gpu l4 --dtype fp8 --mode inference
# !python3 models/ksctl plan --model deepseek-v3 --gpu a100 --dtype bf16   # FP8->bf16 on A100
